# Aula 03 — Chains e Loops de Decisão

**Disciplina:** Deploy de Agentes com LLMs — Pós-Graduação em Machine Learning (FIAP/POSTECH)

Construímos do mais simples ao mais robusto, **sempre com um LLM real** (Claude):
1. **Chain** — pipeline determinístico, montado com o framework **LangChain (LCEL)**
2. **Agente** — loop de decisão (o modelo escolhe as ações)
3. **Observabilidade** — logging do loop
4. **Feedback loop** — auto-correção (Reflexion-lite)

> Requer uma chave de API. Copie `.env.example` para `.env` e preencha `ANTHROPIC_API_KEY`.
> O modelo padrão é `claude-sonnet-4-6` (ajuste com `LLM_MODEL`).

## 0. Setup — carregar o LLM

Carregamos as credenciais do `.env` e instanciamos o modelo **uma única vez**. Todos os
exemplos abaixo reutilizam este `llm`. Use `temperature=0` para respostas reprodutíveis.

In [2]:
# !pip install langchain langchain-anthropic python-dotenv  # se necessário
import os, json
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic

load_dotenv()
assert os.getenv("ANTHROPIC_API_KEY"), "Defina ANTHROPIC_API_KEY no arquivo .env"

# `or` (em vez do default do getenv) garante o padrão também quando LLM_MODEL existe vazio no .env
MODELO = os.getenv("LLM_MODEL") or "claude-sonnet-4-6"
llm = ChatAnthropic(model=MODELO, temperature=0)
print("LLM pronto:", MODELO)

LLM pronto: claude-sonnet-4-6


## Ferramentas externas (sistemas simulados)

Funções que imitam um **sistema externo** de estoque e preços. Não são "dados de LLM":
representam a API/banco que o agente consulta. Em produção, trocaríamos o dicionário por
uma chamada HTTP ou SQL real — a interface para o agente seria a mesma.

In [3]:
ESTOQUE = {"teclado mecânico": 8, "mouse": 0, "monitor": 3}
PRECO   = {"teclado mecânico": 350.0, "mouse": 80.0, "monitor": 1200.0}

def consultar_estoque(produto: str) -> str:
    return str(ESTOQUE.get(produto.lower().strip(), "produto não encontrado"))

def consultar_preco(produto: str) -> str:
    v = PRECO.get(produto.lower().strip())
    return f"{v:.2f}" if v is not None else "produto não encontrado"

FERRAMENTAS = {"consultar_estoque": consultar_estoque, "consultar_preco": consultar_preco}
print("ferramentas:", list(FERRAMENTAS))

ferramentas: ['consultar_estoque', 'consultar_preco']


## 1. Chain — pipeline determinístico com LangChain (LCEL)

Numa **chain**, os passos são **fixos** e QUEM controla a ordem é o nosso código.
Montamos o pipeline com o operador `|` do LangChain (LCEL), encadeando três etapas:

1. **Extrair** o produto da mensagem → LLM com **saída estruturada** (`with_structured_output`).
2. **Consultar** o estoque → função Python encapsulada em `RunnableLambda` (é uma ferramenta).
3. **Formatar** a resposta ao cliente → outro passo de LLM (`prompt | llm | parser`).

A ordem extrair → consultar → formatar é **sempre a mesma**: previsível e fácil de testar.

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from pydantic import BaseModel, Field

# Passo 1 — extrair o produto (LLM com saída estruturada/tipada)
class Produto(BaseModel):
    nome: str = Field(description="nome do produto citado pelo cliente, em minúsculas")

extrair_prompt = ChatPromptTemplate.from_template(
    "Extraia o produto citado nesta mensagem do cliente:\n{texto}"
)
extrair = extrair_prompt | llm.with_structured_output(Produto)

# Passo 2 — consultar o estoque (ferramenta Python como Runnable)
def consultar(p: Produto) -> dict:
    return {"produto": p.nome, "estoque": consultar_estoque(p.nome)}

# Passo 3 — formatar a resposta final (LLM -> texto)
formatar_prompt = ChatPromptTemplate.from_template(
    "Produto: {produto} | Estoque: {estoque}\n"
    "Escreva UMA frase curta e cordial informando a disponibilidade ao cliente.")

# A CHAIN: encadeamento fixo com o operador | (LCEL)
chain = (
    {"texto": RunnablePassthrough()}
    | extrair
    | RunnableLambda(consultar)
    | formatar_prompt | llm | StrOutputParser()
)

print(chain.invoke("Tem o teclado mecânico em estoque?"))
print(chain.invoke("Vocês têm mouse disponível?"))

**O teclado mecânico está disponível em estoque, com 8 unidades prontas para entrega!** 😊
**Mouse** está temporariamente **fora de estoque** no momento. 😊


> **Lição:** a chain é um grafo de passos fixos. O LLM preenche cada etapa, mas o
> **fluxo** é nosso. Barato, testável e previsível — use quando os passos são conhecidos.

## 2. Agente — loop de decisão (política dinâmica)

Agora **não fixamos a ordem**: damos um objetivo e ferramentas, e o **modelo decide**, a
cada iteração, qual ação tomar. A *política* usa **saída estruturada**
(`with_structured_output`, como vimos na Aula 02) para devolver a decisão já **tipada e
validada** num modelo Pydantic `Decisao` — sem extrair JSON do texto na mão (o LangChain
faz isso por nós, e o LLM é obrigado a respeitar o esquema):

- `tipo="acao"` + `ferramenta` + `entrada` → chamar uma ferramenta
- `tipo="responder"` + `conteudo` → concluir

O `max_iter` é a **salvaguarda** contra loops infinitos.

In [5]:
from typing import Literal, Optional
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, Field

# Esquema da decisão: o LLM devolve isto já validado (structured output), sem parsing manual
class Decisao(BaseModel):
    """Próxima ação da política do agente de vendas."""
    tipo: Literal["acao", "responder"] = Field(
        description="'acao' para chamar uma ferramenta; 'responder' para concluir a tarefa")
    ferramenta: Optional[Literal["consultar_estoque", "consultar_preco"]] = Field(
        default=None, description="ferramenta a chamar (quando tipo='acao')")
    entrada: Optional[str] = Field(
        default=None, description="produto passado à ferramenta (quando tipo='acao')")
    conteudo: Optional[str] = Field(
        default=None, description="resposta final ao cliente (quando tipo='responder')")

SISTEMA = """Você é a política de decisão de um agente de vendas. A cada passo, escolha a
próxima ação: chame uma ferramenta (consultar_estoque ou consultar_preco) para coletar o
que ainda falta, ou responda ao cliente quando já tiver estoque E preço. Use as observações
já coletadas para não repetir consultas."""

# with_structured_output força o LLM a devolver um objeto Decisao válido
politica_llm = llm.with_structured_output(Decisao)

def politica(estado: dict) -> Decisao:
    """Pede ao LLM a próxima decisão, já tipada e validada (sem json.loads)."""
    msg = f"Objetivo: {estado['objetivo']}\nObservações: {estado['observacoes']}"
    return politica_llm.invoke([SystemMessage(SISTEMA), HumanMessage(msg)])

def agente_loop(objetivo: str, max_iter: int = 5) -> str:
    estado = {"objetivo": objetivo, "observacoes": []}
    for i in range(1, max_iter + 1):
        d = politica(estado)
        print(f"[iter {i}] {d}")
        if d.tipo == "responder":
            return d.conteudo
        obs = FERRAMENTAS[d.ferramenta](d.entrada)
        estado["observacoes"].append((d.ferramenta, d.entrada, obs))
        print("   obs:", obs)
    return "Limite de iterações atingido (parada de segurança)."

objetivo = "Quero comprar 5 teclados mecânicos. É possível? E qual o valor total?"
print("\nRESPOSTA:", agente_loop(objetivo))

[iter 1] tipo='acao' ferramenta='consultar_estoque' entrada='teclado mecânico' conteudo=None
   obs: 8
[iter 2] tipo='acao' ferramenta='consultar_preco' entrada='teclado mecânico' conteudo=None
   obs: 350.00
[iter 3] tipo='responder' ferramenta=None entrada=None conteudo='Ótima notícia! Já tenho todas as informações que você precisa. Veja o resumo:\n\n🖮 **Teclado Mecânico**\n- **Estoque disponível:** 8 unidades ✅\n- **Preço unitário:** R$ 350,00\n- **Quantidade desejada:** 5 unidades\n\n---\n\n✅ **Sim, é possível realizar a compra!** Temos 8 unidades em estoque, o que é suficiente para atender o seu pedido de 5 unidades.\n\n💰 **Valor total: R$ 1.750,00** (5 × R$ 350,00)\n\nDeseja prosseguir com a compra ou tem alguma outra dúvida? 😊'

RESPOSTA: Ótima notícia! Já tenho todas as informações que você precisa. Veja o resumo:

🖮 **Teclado Mecânico**
- **Estoque disponível:** 8 unidades ✅
- **Preço unitário:** R$ 350,00
- **Quantidade desejada:** 5 unidades

---

✅ **Sim, é possível realiza

## 3. Observabilidade — logging do loop

Um loop que decide sozinho precisa ser **auditável**. Instrumentamos cada iteração
(decisão, ação, observação) com o módulo `logging` — reaproveitando a `politica` acima.

In [6]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logging.getLogger("httpx").setLevel(logging.WARNING)  # silencia os logs HTTP da API
log = logging.getLogger("agente")

def agente_observavel(objetivo: str, max_iter: int = 5) -> str:
    estado = {"objetivo": objetivo, "observacoes": []}
    log.info("início | objetivo=%r | max_iter=%d", objetivo, max_iter)
    for i in range(1, max_iter + 1):
        d = politica(estado)  # reutiliza a política com saída estruturada da seção 2
        log.info("[iter %d] decisao=%s", i, d)
        if d.tipo == "responder":
            log.info("[iter %d] PARADA: agente concluiu", i)
            return d.conteudo
        obs = FERRAMENTAS[d.ferramenta](d.entrada)
        estado["observacoes"].append((d.ferramenta, d.entrada, obs))
        log.info("[iter %d] acao=%s(%r) -> obs=%r", i, d.ferramenta, d.entrada, obs)
    log.warning("limite de iterações atingido SEM conclusão (possível loop)")
    return "Limite de iterações atingido (parada de segurança)."

print("\nRESPOSTA:", agente_observavel("Comprar 5 teclados mecânicos: há estoque e qual o total?"))

2026-10-06 19:18:34,302 | INFO | início | objetivo='Comprar 5 teclados mecânicos: há estoque e qual o total?' | max_iter=5
2026-10-06 19:18:36,741 | INFO | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-10-06 19:18:36,744 | INFO | [iter 1] decisao=tipo='acao' ferramenta='consultar_estoque' entrada='teclado mecânico' conteudo=None
2026-10-06 19:18:36,745 | INFO | [iter 1] acao=consultar_estoque('teclado mecânico') -> obs='8'
2026-10-06 19:18:38,147 | INFO | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-10-06 19:18:38,153 | INFO | [iter 2] decisao=tipo='acao' ferramenta='consultar_preco' entrada='teclado mecânico' conteudo=None
2026-10-06 19:18:38,154 | INFO | [iter 2] acao=consultar_preco('teclado mecânico') -> obs='350.00'
2026-10-06 19:18:41,325 | INFO | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-10-06 19:18:41,326 | INFO | [iter 3] decisao=tipo='responder' ferramenta=None entrada=N


RESPOSTA: Ótimas notícias! Aqui está o resumo da sua consulta para **5 teclados mecânicos**:

- ✅ **Estoque disponível:** 8 unidades (suficiente para a sua compra de 5)
- 💰 **Preço unitário:** R$ 350,00
- 🧾 **Total da compra (5 unidades):** R$ 1.750,00

Posso prosseguir com o pedido? 😊


## 4. Feedback loop — auto-correção a partir de um erro (Reflexion-lite)

Padrão **Reflexion** (Shinn et al., 2023) em versão mínima, agora com um **erro real**: o
agente **age** → a ação **falha** com uma exceção → ele **lê a mensagem de erro** e **tenta
de novo** usando esse erro como contexto. Sem retreinamento — só reorganizamos o prompt.

Cenário: o catálogo é consultado por **SKU** (código), e não pelo nome. O agente **não
conhece os SKUs** de antemão; na 1ª tentativa ele chuta, a ferramenta `consultar_produto`
**lança uma exceção** listando os SKUs válidos, e o agente se corrige na tentativa seguinte.
É o mesmo mecanismo dos agentes que escrevem código, rodam e leem o erro para se ajustar.

In [7]:
# Catálogo consultável por SKU (código). Os NOMES não são aceitos pela ferramenta.
CATALOGO = {
    "TEC-001": {"nome": "teclado mecânico", "preco": 350.0, "estoque": 8},
    "MOU-002": {"nome": "mouse", "preco": 80.0, "estoque": 0},
    "MON-003": {"nome": "monitor", "preco": 1200.0, "estoque": 3},
}

def consultar_produto(sku: str) -> dict:
    """Consulta um produto pelo SKU. LANÇA KeyError (com os SKUs válidos) se não existir."""
    chave = sku.strip().upper()
    if chave not in CATALOGO:
        validos = ", ".join(f"{k} ({v['nome']})" for k, v in CATALOGO.items())
        raise KeyError(f"SKU '{sku}' não encontrado. SKUs válidos: {validos}")
    return CATALOGO[chave]

# O LLM decide qual SKU consultar (saída estruturada)
class Chamada(BaseModel):
    """Argumento da ferramenta consultar_produto."""
    sku: str = Field(description="o SKU (código) do produto a consultar")

gerar_chamada = llm.with_structured_output(Chamada)

INSTRUCAO = ("Você consulta um catálogo pela ferramenta consultar_produto(sku). Decida o SKU "
             "a usar. Se uma tentativa anterior falhou, leia a mensagem de erro e corrija o SKU.")

def agente_reflexivo(objetivo: str, max_tentativas: int = 3):
    historico = ""  # acumula os erros anteriores -> vira contexto (a "reflexão")
    for t in range(1, max_tentativas + 1):
        # 1. GERAR a próxima chamada (com o histórico de erros, se houver)
        msg = f"Objetivo: {objetivo}{historico}"
        chamada = gerar_chamada.invoke([SystemMessage(INSTRUCAO), HumanMessage(msg)])
        print(f"[tentativa {t}] consultar_produto(sku={chamada.sku!r})")
        # 2. EXECUTAR a ação e CAPTURAR a exceção
        try:
            produto = consultar_produto(chamada.sku)
        except KeyError as erro:
            critica = erro.args[0]
            print(f"             ✗ ERRO: {critica}")
            historico += f"\nA tentativa com sku={chamada.sku!r} falhou: {critica}"  # 3. REFLEXÃO
            continue
        # 4. SUCESSO
        print(f"             ✓ OK: {produto['nome']} — R$ {produto['preco']:.2f}, "
              f"estoque {produto['estoque']}")
        return produto
    print("             (esgotou as tentativas sem sucesso)")
    return None

print("\nRESULTADO:", agente_reflexivo("Descubra o preço do teclado mecânico."))

2026-10-06 19:20:40,548 | INFO | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


[tentativa 1] consultar_produto(sku='teclado-mecanico')
             ✗ ERRO: SKU 'teclado-mecanico' não encontrado. SKUs válidos: TEC-001 (teclado mecânico), MOU-002 (mouse), MON-003 (monitor)


2026-10-06 19:20:41,825 | INFO | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


[tentativa 2] consultar_produto(sku='TEC-001')
             ✓ OK: teclado mecânico — R$ 350.00, estoque 8

RESULTADO: {'nome': 'teclado mecânico', 'preco': 350.0, 'estoque': 8}


> Repare na sequência **agir → falhar → ler o erro → corrigir**: a exceção da ferramenta
> vira contexto da próxima tentativa, e o agente sai do erro sozinho. O `max_tentativas`
> é a salvaguarda — sem ele, um erro persistente viraria loop infinito.

## Conclusão

Saímos da **chain** (controle do fluxo é nosso, montada com LCEL do LangChain) para o
**agente** (o modelo decide no loop de decisão). **Logging**, **limite de iterações** e
**feedback loops** tornam essa autonomia **segura e auditável** — a base para levar agentes
à produção. Tudo rodando com um **LLM real**, sem simulações da política.